## Feature Engineering

### Load Data

In [1]:
import numpy as np
import pandas as pd
import os
from pathlib import Path

# Read raw data
master_dir = os.path.dirname(os.getcwd())

cleaned_reviews = os.path.join(master_dir, "data", "cleaned_data", "cleaned_reviews.csv")

df_cleaned = pd.read_csv(cleaned_reviews)

In [2]:
# Add Review_ID column

df_cleaned.insert(
    0,
    "Review_ID",
    ["R{:06d}".format(i) for i in range(1, len(df_cleaned) + 1)]
)

df_cleaned.head(10)

,Review_ID,Review,Rating,Lemmatized_Tokens,Review_lemmatized,Platform,Restaurant,Sentiment_Label
0,R000001,came here for the high tea. great service espe...,4.0,"['come', 'high', 'tea', '.', 'great', 'service...",come high tea . great service especially mr. j...,Google,Cuisines Restaurant,Positive
1,R000002,"5 stars for the service, even though some of t...",2.0,"['5', 'star', 'service', ',', 'even', 'though'...","5 star service , even though staff need train ...",Google,Cuisines Restaurant,Negative
2,R000003,"hi, thank you for your service. but! i feel so...",1.0,"['hi', ',', 'thank', 'service', '.', 'but', '!...","hi , thank service . but ! feel so sorry food ...",Google,Cuisines Restaurant,Negative
3,R000004,i have the worse buffer dinner ever so far. th...,1.0,"['bad', 'buffer', 'dinner', 'ever', 'so', 'far...",bad buffer dinner ever so far . spread so smal...,Google,Cuisines Restaurant,Negative
4,R000005,"that is are known 5 elmark "" 9h72 "" & kdk "" 3 ...",5.0,"['know', '5', 'elmark', '``', '9h72', '``', '&...",know 5 elmark `` 9h72 `` & kdk `` 3 k14y9 & 1 ...,Google,Cuisines Restaurant,Positive
5,R000006,i just came back from there. 2 adults and 4 yo...,2.0,"['come', 'back', '.', '2', 'adult', '4', 'youn...",come back . 2 adult 4 young child . look exclu...,Google,Cuisines Restaurant,Negative
6,R000007,restaurant looks nice but taste is bad. i had ...,2.0,"['restaurant', 'look', 'nice', 'but', 'taste',...",restaurant look nice but taste bad . bbq buffe...,Google,Cuisines Restaurant,Negative
7,R000008,"pros: ambience is great with lake view, good a...",4.0,"['pro', ':', 'ambience', 'great', 'lake', 'vie...","pro : ambience great lake view , good air cond...",Google,Cuisines Restaurant,Positive
8,R000009,we went to this place after reviews on tripadv...,1.0,"['go', 'place', 'review', 'tripadvisor', 'tota...",go place review tripadvisor totally disappoint...,Google,Cuisines Restaurant,Negative
9,R000010,"the restaurant is located inside the hotel, th...",4.0,"['restaurant', 'locate', 'inside', 'hotel', ',...","restaurant locate inside hotel , day come many...",Google,Cuisines Restaurant,Positive


### TF-IDF Vectorization

In [3]:
# check is that any missing value in the lemmatized review column
print(df_cleaned["Review_lemmatized"].isna().sum())

18


In [4]:
df_cleaned[df_cleaned["Review_lemmatized"].isna()][
    ["Review", "Review_lemmatized"]
].head()

,Review,Review_lemmatized
12416,just because,NaN
22957,i,NaN
24837,y you,NaN
28437,up,NaN
38554,we are here,NaN


In [5]:
# remove the rows with missing values in the lemmatized review column
df_cleaned = df_cleaned.dropna(subset=["Review_lemmatized"])

In [6]:
from sklearn.feature_extraction.text import TfidfVectorizer

tfidf_vectorizer = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=5,
    max_df=0.9
)

tfidf_matrix  = tfidf_vectorizer.fit_transform(df_cleaned["Review_lemmatized"])

In [7]:
# check the shape of the TF-IDF matrix
print("TF-IDF shape:", tfidf_matrix.shape)

TF-IDF shape: (361762, 5000)


In [8]:
feature_names = tfidf_vectorizer.get_feature_names_out()

print("Total vocabulary size:", len(feature_names))

Total vocabulary size: 5000


In [9]:
mean_tfidf = np.asarray(tfidf_matrix.mean(axis=0)).flatten()

top_indices = mean_tfidf.argsort()[::-1][:20]

top_tfidf_features = pd.DataFrame({
    "Feature": feature_names[top_indices],
    "Mean_TFIDF": mean_tfidf[top_indices]
})

top_tfidf_features

,Feature,Mean_TFIDF
0,food,0.046786
1,good,0.042766
2,nice,0.029897
3,not,0.027713
4,very,0.026251
5,place,0.025511
6,service,0.025001
7,great,0.024702
8,but,0.022134
9,delicious,0.018326


### Unigrams & Bigrams

In [10]:
unigrams = [
    feature for feature in feature_names
    if len(feature.split()) == 1
]

bigrams = [
    feature for feature in feature_names
    if len(feature.split()) == 2
]

print("Number of unigrams:", len(unigrams))
print("Number of bigrams:", len(bigrams))

Number of unigrams: 2659
Number of bigrams: 2341


In [11]:
# get the top unigrams
unigram_indices = [
    i for i, feature in enumerate(feature_names)
    if len(feature.split()) == 1
]

top_unigram_indices = sorted(
    unigram_indices,
    key=lambda i: mean_tfidf[i],
    reverse=True
)[:20]

top_unigrams = pd.DataFrame({
    "Feature": feature_names[top_unigram_indices],
    "Mean_TFIDF": mean_tfidf[top_unigram_indices]
})

top_unigrams

,Feature,Mean_TFIDF
0,food,0.046786
1,good,0.042766
2,nice,0.029897
3,not,0.027713
4,very,0.026251
5,place,0.025511
6,service,0.025001
7,great,0.024702
8,but,0.022134
9,delicious,0.018326


In [12]:
# get the top bigrams
bigram_indices = [
    i for i, feature in enumerate(feature_names)
    if len(feature.split()) == 2
]

top_bigram_indices = sorted(
    bigram_indices,
    key=lambda i: mean_tfidf[i],
    reverse=True
)[:20]

top_bigrams = pd.DataFrame({
    "Feature": feature_names[top_bigram_indices],
    "Mean_TFIDF": mean_tfidf[top_bigram_indices]
})

top_bigrams

,Feature,Mean_TFIDF
0,show less,0.013878
1,good food,0.012530
2,nice food,0.007935
3,food good,0.007918
4,very good,0.007453
5,good service,0.007381
6,great food,0.006407
7,nice place,0.006119
8,very nice,0.005328
9,friendly staff,0.004872


### Export Feature Data

In [13]:
# Project directory
project_dir = os.path.dirname(os.getcwd())

feature_info = pd.DataFrame({
    "Feature_Index": range(len(feature_names)),
    "Feature": feature_names,
    "Mean_TFIDF": mean_tfidf
})


feature_info.to_csv(
    os.path.join(
        project_dir,
        "data",
        "processed_data",
        "tfidf_features.csv"
    ),
    index=False
)

### Save TF-IDF Metric

In [14]:
from scipy.sparse import save_npz

tfidf_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "tfidf_matrix.npz"
)

save_npz(tfidf_path, tfidf_matrix)

print("TF-IDF matrix saved to:", tfidf_path)

TF-IDF matrix saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\tfidf_matrix.npz


### Save Feature Engineered Reviews

In [15]:
feature_data_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "feature_engineered_reviews.csv"
)

df_cleaned.to_csv(
    feature_data_path,
    index=False
)

print("Feature-engineered dataset saved to:", feature_data_path)

Feature-engineered dataset saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\feature_engineered_reviews.csv


### Save TF-IDF Vectorizer

In [16]:
import joblib

vectorizer_path = os.path.join(
    project_dir,
    "data",
    "processed_data",
    "tfidf_vectorizer.pkl"
)

joblib.dump(
    tfidf_vectorizer,
    vectorizer_path
)

print("TF-IDF vectorizer saved to:", vectorizer_path)

TF-IDF vectorizer saved to: c:\Users\user\Downloads\social_computing_assignment\data\processed_data\tfidf_vectorizer.pkl
